# EV Charging Demand - end-to-end walkthrough
Runs the same modules used by `python -m src.pipeline`, step by step. (Synthetic stand-in data unless `data/raw/charging_data.csv` is provided.)

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src.pipeline import load_or_generate_raw
from src.data_cleaning import describe_data, clean_data
from src.feature_engineering import build_features, split_chronological
raw, synthetic = load_or_generate_raw(72000)
describe_data(raw)

## Cleaning

In [ ]:
clean, report = clean_data(raw)
report

## Feature engineering and EDA

In [ ]:
feats = build_features(clean)
from src.eda import run_eda
run_eda(feats)['findings']

## Regression

In [ ]:
train, test = split_chronological(feats)
from src.regression import train_regression
reg = train_regression(train, test, fast=True)
reg['results']

## Classification (median threshold, imbalance check)

In [ ]:
from src.classification import train_classification
clf = train_classification(train, test, fast=True)
print(clf['imbalance'])
clf['results']

## Explainability and forecasting

In [ ]:
from src.explainability import run_explainability
print(run_explainability(reg['best_pipeline'], clf['best_pipeline'], test, train))
from src.forecasting import run_forecasting
run_forecasting(feats, test_days=30)['results']